# 09 Layoffs

Both `GET /api/{market}/technical-strategies/cohorts` and `GET /api/us/layoffs/latest` are member research routes (beta).

This notebook fetches the US layoff feed once. The filing table and historical study tables use that same response.

The source date and data status appear above each table. HTTP 503 with `LAYOFF_FEED_UNAVAILABLE` shows **Feed unavailable**, not an error.

The study contains historical statistics. Its event counts, comparison groups, and limitations describe the supplied evidence. Research data only, not investment advice.

Free access grants no redistribution rights. **Do not republish the data you fetch.**

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'notebooks' / '_shared.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'notebooks'))
sys.path.insert(0, str(ROOT / 'python' / 'src'))
from _shared import fetch_payload, get_api_key, make_client, show_panel
client, catalogue = make_client()
api_key = get_api_key()
layoffs_payload = fetch_payload(
    '09 Layoffs — us',
    lambda: client.member_layoffs(api_key),
)

In [ ]:
WATCH_COLUMNS = {
    'company': 'The company named in the SEC filing.',
    'ticker': 'The current US ticker when the feed supplies one.',
    'sector': 'The feed sector: tech or finance.',
    'filing_date': 'The date of the SEC filing.',
    'filing_url': 'The SEC URL for the source filing.',
    'state': 'The feed classification from the distress checks.',
    'price_collapse': 'Whether the historical price fall passed the stated distress threshold.',
    'bankruptcy_or_default_filing': 'Whether the feed found a relevant bankruptcy or default filing.',
    'losing_money_and_burning_cash': 'Whether the latest annual statement met both stated loss and cash criteria.',
    'prior_12m_return': 'The historical price change for the 12 months before the filing, excluding dividends.',
}
def watch_rows(payload):
    return [
        {
            'company': row.get('company'),
            'ticker': row.get('ticker'),
            'sector': row.get('sector'),
            'filing_date': row.get('filing_date'),
            'filing_url': row.get('filing_url'),
            'state': row.get('state'),
            'price_collapse': (row.get('checks') or {}).get('price_collapse'),
            'bankruptcy_or_default_filing': (row.get('checks') or {}).get('bankruptcy_or_default_filing'),
            'losing_money_and_burning_cash': (row.get('checks') or {}).get('losing_money_and_burning_cash'),
            'prior_12m_return': row.get('prior_12m_return'),
        }
        for row in payload.get('watch') or []
    ]
if layoffs_payload is not None:
    watch_frame = show_panel(
        '09 Layoff watch — us',
        layoffs_payload,
        watch_rows(layoffs_payload),
        WATCH_COLUMNS,
        source_date=layoffs_payload.get('data_through'),
        status='available' if layoffs_payload.get('ok') is True else 'not supplied',
        message=f"Published at: {layoffs_payload.get('published_at', 'not supplied')}",
    )

In [ ]:
STUDY_COLUMNS = {
    'group': 'The distress and sector group used for historical comparison.',
    'horizon_sessions': 'The number of trading sessions after the filing.',
    'n': 'The number of historical events in this group and horizon.',
    'mean': 'The mean historical price change relative to industry peers, before costs and dividends.',
    'median': 'The median historical price change relative to industry peers, before costs and dividends.',
    'share_positive': 'The fraction of historical events with a positive industry-relative price change.',
    'se_clustered_month': 'The standard error clustered by month.',
    't': 'The historical mean divided by its clustered standard error.',
    'clusters': 'The number of month clusters used for the standard error.',
}
def study_rows(payload):
    rows = []
    for group, horizons in ((payload.get('study') or {}).get('groups') or {}).items():
        for horizon, statistics in horizons.items():
            rows.append({
                'group': group,
                'horizon_sessions': horizon,
                'n': statistics.get('n'),
                'mean': statistics.get('mean'),
                'median': statistics.get('median'),
                'share_positive': statistics.get('share_positive'),
                'se_clustered_month': statistics.get('se_clustered_month'),
                't': statistics.get('t'),
                'clusters': statistics.get('clusters'),
            })
    return rows
if layoffs_payload is not None:
    study = layoffs_payload.get('study') or {}
    message = (
        f"Primary comparison: {study.get('primary', 'not supplied')}; "
        f"SEC events: {study.get('events_layoff_8k', 'not supplied')}; "
        f"events after 90-day deduplication: {study.get('events_after_90d_dedup', 'not supplied')}; "
        f"published at: {layoffs_payload.get('published_at', 'not supplied')}"
    )
    study_frame = show_panel(
        '09 Layoff historical study — us',
        layoffs_payload,
        study_rows(layoffs_payload),
        STUDY_COLUMNS,
        source_date=layoffs_payload.get('data_through'),
        status='available' if layoffs_payload.get('ok') is True else 'not supplied',
        message=message,
    )
    print('Method:', layoffs_payload.get('method', 'not supplied'))
    print('Study limitations:')
    for limitation in study.get('limitations') or ['not supplied']:
        print(f'- {limitation}')
    print('Disclaimer:', layoffs_payload.get('disclaimer', 'not supplied'))